# 📊 Strategy 1 (Causal Graph) End-to-End Pipeline & Backtest (Live)

Ce notebook présente le pipeline complet de traitement des données de la **Stratégie 1 (Graphe Causal)** en temps réel.

Le pipeline de nettoyage (Usability Funnel), le calcul des features par minute, et la simulation de backtest sont **entièrement recalculés en direct** à partir des bases de données SQLite brutes.
Seuls les paramètres de calibration structurels (Z-scores optimaux, coefficients et statistiques de normalisation) et les liens du graphe causal (`causal_edges_v2.parquet`) sont précalculés et chargés.

---

### Étapes du Pipeline :
1. **Configuration & Chargement Direct** : Chargement des bases brutes SQLite sur la période choisie.
2. **Nettoyage (Usability Funnel) Live** : Filtrage et audit des marchés Polymarket.
3. **Calcul des Features Live** : Agrégation dense à la minute, VWAP proxy et calcul de l'incertitude (entropie de Shannon).
4. **Catégorisation & Liens Causaux** : Attribution aux catégories et chargement des relations causales calibrées.
5. **Simulation de Backtest Live** : Exécution de la stratégie de trading sur les features et prix calculés.
6. **Visualisation des Performances & Risques** : Analyse premium des résultats (PnL, Drawdown, Sharpe/Vol, Stacked weights, Heatmap, Distribution).


In [ ]:
import sys
import os
import json
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from IPython.display import display, HTML

# Configurer le chemin pour importer les modules du pipeline et du backtest
sys.path.append('.')
sys.path.append('..')

# Styles et palettes premium
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "figure.figsize": (15, 6),
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Arial", "Helvetica"],
    "axes.edgecolor": "#E5E7EB",
    "axes.linewidth": 0.8,
    "grid.color": "#F3F4F6",
    "grid.linestyle": "--",
    "grid.alpha": 0.7,
})

COLORS = {
    'primary': '#3B82F6',       # Bleu électrique moderne
    'success': '#10B981',       # Vert émeraude (si profitable)
    'danger': '#EF4444',        # Rouge corail (si perte)
    'warning': '#F59E0B',       # Orange/Ambre
    'drawdown': '#F43F5E',      # Rose/Rouge vif pour le drawdown
    'accent': '#8B5CF6',        # Violet
    'gray': '#9CA3AF',          # Gris
    'light_gray': '#E5E7EB',
}


## 1. Configuration & Chargement Direct depuis SQLite
Vous pouvez modifier les chemins de base de données brutes ou la date de début de l'analyse ci-dessous. Le notebook se charge de filtrer et charger les données directement depuis SQLite.


In [ ]:
# ── Configuration utilisateur des bases de données ────────────────────────
DB_PATH = "../Donnees_Brutes/polymarket_student_train_2025.sqlite3"
DB_STOCKS_PATH = "../Donnees_Brutes/sp100_2025_1min_bars.sqlite3"
START_DATE = "2025-11-01"  # Début de la période d'analyse (Période de Test)

# Si exécuté depuis un autre dossier
if not os.path.exists(DB_PATH):
    DB_PATH = "Donnees_Brutes/polymarket_student_train_2025.sqlite3"
    DB_STOCKS_PATH = "Donnees_Brutes/sp100_2025_1min_bars.sqlite3"

print("1. Chargement des paramètres de calibration de la Stratégie 1...")
with open("STRATEGIES/calibration_strat1.json") as f:
    calib = json.load(f)

z_long = calib.get("z_long", calib.get("optimal_z", 7.0))
z_short = calib.get("z_short", z_long + calib.get("asymmetric_short_penalty", 0.5))
print(f"  ✓ Seuil Z-score d'entrée Long : {z_long:.2f}")
print(f"  ✓ Seuil Z-score d'entrée Short : {z_short:.2f}")
print(f"  ✓ Facteur d'échelle (Scaling factor) : {calib.get('scaling_factor', 0.5):.4f}")

print("\n2. Connexion et requête SQLite (chargement des tables Polymarket)...")
conn = sqlite3.connect(DB_PATH)
df_markets = pd.read_sql("SELECT * FROM market", conn)
df_token = pd.read_sql("SELECT * FROM token_outcome", conn)
df_market_tag = pd.read_sql("SELECT * FROM market_tag", conn)
df_filter_tag = pd.read_sql("SELECT * FROM selected_filter_tag", conn)

# Requête filtrée par date sur trade_price_1m pour optimiser la mémoire
query_bars = f"SELECT * FROM trade_price_1m WHERE minute_ts >= '{START_DATE}'"
df_bars = pd.read_sql(query_bars, conn)
conn.close()

# Formater les colonnes temporelles
df_markets["start_date"] = pd.to_datetime(df_markets["start_date"], utc=True, errors="coerce")
df_markets["end_date"] = pd.to_datetime(df_markets["end_date"], utc=True, errors="coerce")
df_bars["minute_ts"] = pd.to_datetime(df_bars["minute_ts"], utc=True, errors="coerce")

# Convertir les prix textuels en float
price_cols = ["open_price", "high_price", "low_price", "close_price", "volume_shares_1m", "notional_usdc_1m"]
for col in price_cols:
    if col in df_bars.columns:
        df_bars[col] = pd.to_numeric(df_bars[col], errors="coerce")

print(f"  ✓ {df_markets.shape[0]} marchés chargés.")
print(f"  ✓ {df_bars.shape[0]:,} barres de prix 1-min chargées pour la période >= {START_DATE}.")

print("\n3. Chargement des prix des actions S&P 100 de marché...")
from backtest_engine import load_sp100_bars
prices_sp100 = load_sp100_bars(tickers=['AAPL', 'MSFT', 'AMZN', 'GOOGL'], start=START_DATE, resample='D')
print(f"  ✓ Cours journaliers chargés : {prices_sp100.shape[0]} jours.")

# Visualisation rapide
fig, ax = plt.subplots(figsize=(12, 4))
for ticker in prices_sp100.columns:
    ax.plot(prices_sp100.index, prices_sp100[ticker], label=ticker, linewidth=2)
ax.set_title("Évolution des cours journaliers (Période sélectionnée)", fontsize=12, fontweight='bold')
ax.set_ylabel("Prix ($)")
ax.legend(frameon=True, edgecolor='#E5E7EB')
plt.tight_layout()
plt.show()


## 2. Nettoyage et Audit en Direct (Usability Funnel Live)
Nous appliquons l'entonnoir d'attrition pour filtrer les marchés exploitables. Le filtre sémantique charge le cache précalculé des clusters sémantiques. En cas de nouveaux marchés non répertoriés, il interroge l'API DeepSeek, et renvoie une erreur en cas d'échec.


In [ ]:
from pipeline.step1_audit import map_and_audit_database
from pipeline.step2_funnel import apply_usability_funnel
from pipeline.config import FunnelConfig

print("1. Exécution de l'audit microstructure (Etape 1)...")
audit_report, per_market_raw = map_and_audit_database(df_markets, df_bars, df_token, df_market_tag)

print("\n2. Application de l'entonnoir d'attrition (Etape 2)...")
cfg = FunnelConfig()
df_usable_markets, df_usable_bars, attrition = apply_usability_funnel(
    df_markets, df_bars, df_token, df_filter_tag, df_market_tag, per_market_raw, cfg
)

print(f"\n✓ Nettoyage terminé. Marchés exploitables restants : {df_usable_markets.shape[0]} / {df_markets.shape[0]}")

# Visualisation de l'entonnoir d'attrition
labels  = ["Raw Data"] + [f"[{s['step']}] {s['label']}" for s in attrition]
n_vals  = [attrition[0]["n_before"]] + [s["n_after"] for s in attrition]
n_total = n_vals[0]

fig, ax = plt.subplots(figsize=(13, 5))
y_pos = list(range(len(labels)))
bar_colors = [COLORS['primary']] + [
    COLORS['success'] if i == len(attrition) - 1 else COLORS['warning']
    for i in range(len(attrition))
]
bars = ax.barh(y_pos, n_vals, color=bar_colors, edgecolor="white", linewidth=0.8, height=0.6)

for bar, val in zip(bars, n_vals):
    pct = val / n_total * 100
    ax.text(bar.get_width() + n_total * 0.005, bar.get_y() + bar.get_height() / 2,
            f"{val:,} ({pct:.1f}%)", va="center", ha="left", fontsize=9, fontweight='semibold')

ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=9.5)
ax.invert_yaxis()
ax.set_xlabel("Nombre de marchés restants", fontweight='semibold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.set_xlim(0, n_total * 1.2)
ax.set_title("Filtres de Sélection Polymarket (Recalculé en Direct)", fontsize=13, fontweight="bold", pad=12)
plt.tight_layout()
plt.show()


## 3. Calcul des Features Avancées en Direct
Nous calculons maintenant les indicateurs agrégés par minute pour chaque catégorie de marché éligible, en utilisant les fichiers de correspondance statiques pour les catégories LLM et les polarités d'actifs.


In [ ]:
print("1. Chargement des catégories statiques et polarités...")
path_cats = "../Donnees_Netoyees/market_categories.parquet"
path_pols = "../Donnees_Netoyees/market_polarities.parquet"
if not os.path.exists(path_cats):
    path_cats = "Donnees_Netoyees/market_categories.parquet"
    path_pols = "Donnees_Netoyees/market_polarities.parquet"
    
df_categories = pd.read_parquet(path_cats, columns=["market_id", "llm_category"])
df_polarities = pd.read_parquet(path_pols, columns=["market_id", "polarity"])

from pipeline.dynamic_classifier import classify_and_polarize_new_markets
df_categories, df_polarities = classify_and_polarize_new_markets(df_usable_markets, df_categories, df_polarities)

def compute_features_live(df_usable_bars, df_usable_markets, df_categories, df_polarities):
    df_market_map = df_usable_markets[["market_id", "condition_id"]].copy()
    df_market_map["market_id"] = df_market_map["market_id"].astype(int)
    df_categories["market_id"] = df_categories["market_id"].astype(int)
    df_polarities["market_id"] = df_polarities["market_id"].astype(int)
    
    mapping = df_market_map.merge(df_categories, on="market_id", how="left")
    mapping = mapping.merge(df_polarities, on="market_id", how="left")
    mapping = mapping[["condition_id", "llm_category", "polarity"]].drop_duplicates()
    
    df_trades = df_usable_bars.merge(mapping, on="condition_id", how="left")
    df_trades["llm_category"] = df_trades["llm_category"].fillna("Other")
    df_trades["polarity"] = df_trades["polarity"].fillna(0).astype(int)
    
    # Calculs vectorisés
    df_trades["signed_close_price"] = np.where(df_trades["polarity"] != 0, df_trades["close_price"] * df_trades["polarity"], np.nan)
    df_trades["price_notional_signed"] = df_trades["signed_close_price"] * df_trades["notional_usdc_1m"]
    df_trades["notional_usdc_1m_signed"] = np.where(df_trades["polarity"] != 0, df_trades["notional_usdc_1m"], np.nan)
    df_trades["price_notional"] = df_trades["close_price"] * df_trades["notional_usdc_1m"]
    df_trades["notional_squared"] = df_trades["notional_usdc_1m"] ** 2
    df_trades["high_minus_low"] = df_trades["high_price"] - df_trades["low_price"]
    
    eps = 1e-9
    p = df_trades["close_price"].clip(0.0, 1.0)
    df_trades["shannon_entropy_m"] = -(p * np.log(p + eps) + (1.0 - p) * np.log(1.0 - p + eps))
    
    # Agréger par minute et par catégorie
    df_grp = df_trades.groupby(["minute_ts", "llm_category"])
    df_basic = df_grp.agg({
        "signed_close_price": ["mean", "std", "max", "min", "median"],
        "price_notional_signed": "sum",
        "notional_usdc_1m_signed": "sum",
        "notional_usdc_1m": "sum",
        "notional_squared": "sum",
        "volume_shares_1m": "sum",
        "trades_count_1m": "sum",
        "condition_id": "nunique",
        "high_minus_low": "mean",
        "shannon_entropy_m": "mean",
    })
    
    df_basic.columns = [f"{col[0]}_{col[1]}" for col in df_basic.columns]
    df_basic["price_skew"] = df_grp["signed_close_price"].skew()
    df_basic["price_kurt"] = df_grp["signed_close_price"].apply(lambda x: x.kurt())
    df_basic["price_q25"] = df_grp["signed_close_price"].quantile(0.25)
    df_basic["price_q75"] = df_grp["signed_close_price"].quantile(0.75)
    df_basic = df_basic.reset_index()
    
    df_basic["price_weighted_mean"] = df_basic["price_notional_signed_sum"] / df_basic["notional_usdc_1m_signed_sum"]
    df_basic["price_weighted_mean"] = df_basic["price_weighted_mean"].fillna(df_basic["signed_close_price_mean"])
    df_basic["hhi_volume"] = df_basic["notional_squared_sum"] / (df_basic["notional_usdc_1m_sum" ] ** 2)
    df_basic["hhi_volume"] = df_basic["hhi_volume"].fillna(1.0).clip(0.0, 1.0)
    
    df_basic = df_basic.rename(columns={
        "signed_close_price_mean": "price_mean",
        "signed_close_price_std": "price_std",
        "signed_close_price_max": "price_max",
        "signed_close_price_min": "price_min",
        "signed_close_price_median": "price_median",
        "volume_shares_1m_sum": "volume_shares_total",
        "notional_usdc_1m_sum": "notional_usdc_total",
        "trades_count_1m_sum": "trades_count_total",
        "condition_id_nunique": "active_markets_count",
        "high_minus_low_mean": "spread_mean",
        "shannon_entropy_m_mean": "shannon_entropy",
    })
    df_basic = df_basic.drop(columns=["price_notional_signed_sum", "notional_usdc_1m_signed_sum", "notional_squared_sum"])
    
    # Re-indexation sur une grille temporelle dense 1-minute
    min_ts = df_basic["minute_ts"].min()
    max_ts = df_basic["minute_ts"].max()
    categories = df_basic["llm_category"].unique()
    full_timeline = pd.date_range(start=min_ts, end=max_ts, freq="min", name="minute_ts")
    mux = pd.MultiIndex.from_product([full_timeline, categories], names=["minute_ts", "llm_category"])
    df_dense = df_basic.set_index(["minute_ts", "llm_category"]).reindex(mux).reset_index()
    
    df_dense["active_markets_count"] = df_dense["active_markets_count"].fillna(0).astype(int)
    df_dense["volume_shares_total"] = df_dense["volume_shares_total"].fillna(0.0)
    df_dense["notional_usdc_total"] = df_dense["notional_usdc_total"].fillna(0.0)
    df_dense["trades_count_total"] = df_dense["trades_count_total"].fillna(0).astype(int)
    
    # Calculs glissants par catégorie (rendements, volumes cumulés...)
    category_dfs = []
    for cat in categories:
        df_cat = df_dense[df_dense["llm_category"] == cat].sort_values("minute_ts").copy()
        df_cat["return_1m"] = df_cat["price_weighted_mean"].pct_change(1, fill_method=None)
        df_cat["return_5m"] = df_cat["price_weighted_mean"].pct_change(5, fill_method=None)
        df_cat["vol_roll_5m"] = df_cat["notional_usdc_total"].rolling(5, min_periods=1).sum()
        df_cat["vol_roll_1h"] = df_cat["notional_usdc_total"].rolling(60, min_periods=1).sum()
        df_cat["trades_roll_5m"] = df_cat["trades_count_total"].rolling(5, min_periods=1).sum()
        df_cat["trades_roll_1h"] = df_cat["trades_count_total"].rolling(60, min_periods=1).sum()
        df_cat["entropy_change_5m"] = df_cat["shannon_entropy"].diff(5)
        category_dfs.append(df_cat)
        
    df_final_long = pd.concat(category_dfs, ignore_index=True)
    
    # Pivoter au format large (wide)
    df_pivot = df_final_long.pivot(index="minute_ts", columns="llm_category")
    df_pivot.columns = [f"{col[1]}_{col[0]}" for col in df_pivot.columns]
    df_final_wide = df_pivot.reset_index()
    return df_final_wide

print("2. Calcul en cours des features (grille 1m)... (peut prendre ~10-15s)")
df_features = compute_features_live(df_usable_bars, df_usable_markets, df_categories, df_polarities)
df_features["minute_ts"] = pd.to_datetime(df_features["minute_ts"], utc=True)
df_features = df_features.set_index("minute_ts")
print(f"✓ Features calculées : {df_features.shape[0]} lignes × {df_features.shape[1]} colonnes.")

# Visualisation sur une catégorie spécifique
sample_category = "Crypto Prices"
price_col = f"{sample_category}_price_weighted_mean"
entropy_col = f"{sample_category}_shannon_entropy"

if price_col in df_features.columns:
    df_resampled = df_features[[price_col, entropy_col]].resample('15min').mean()
    price_diff = df_resampled[price_col].diff().fillna(0)
    z_score = (price_diff - price_diff.mean()) / price_diff.std()
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
    ax1.plot(df_resampled.index, df_resampled[price_col], color=COLORS['primary'], label="Prix Moyen Pondéré Live")
    ax1.set_ylabel("Prix Moyen Pondéré")
    ax1.set_title(f"Features calculées en direct — Catégorie: '{sample_category}'", fontsize=13, fontweight='bold')
    ax1.legend(loc='upper left')
    
    ax2.plot(z_score.index, z_score, color=COLORS['accent'], label="Z-score glissant (15m)")
    ax2.axhline(0, color='grey', linestyle='--', linewidth=0.8)
    ax2.axhline(z_long, color=COLORS['danger'], linestyle=':', label=f"Seuil Long (+{z_long:.2f})")
    ax2.axhline(-z_short, color=COLORS['danger'], linestyle=':', label=f"Seuil Short (-{z_short:.2f})")
    ax2.set_ylabel("Z-score")
    ax2.set_xlabel("Date")
    ax2.legend(loc='upper left')
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%d %b\n%Y'))
    plt.tight_layout()
    plt.show()
else:
    print("La catégorie d'exemple n'a pas de colonnes de prix.")


## 4. Relations Causales Précalculées
Nous chargeons ici le graphe causal qui sert de passerelle entre nos features Polymarket et l'univers d'actifs TradFi S&P 100.


In [ ]:
print("Top 15 Causal Links in Strategy 1 (Causal Graph):")
display(edges_df.sort_values(by="weight", key=abs, ascending=False).head(15))

# Visualisation
top_edges = edges_df.sort_values(by="weight", key=abs, ascending=False).head(20).copy()
top_edges["link"] = top_edges["source"] + " → " + top_edges["target"]
colors = [COLORS['success'] if w > 0 else COLORS['danger'] for w in top_edges["weight"]]

fig, ax = plt.subplots(figsize=(12, 5.5))
bars = ax.barh(top_edges["link"], top_edges["weight"], color=colors, edgecolor="white", linewidth=0.5)
ax.axvline(0, color='grey', linestyle='-', linewidth=0.8)
ax.invert_yaxis()
ax.set_xlabel("Poids du Lien Causal", fontweight='semibold')
ax.set_title("Top 20 des Relations Causales (Catégorie Polymarket → Stock S&P 100)", fontsize=13, fontweight='bold', pad=12)
ax.grid(True, linestyle=":", alpha=0.5)

handles = [
    mpatches.Patch(color=COLORS['success'], label="Poids Positif (+)"),
    mpatches.Patch(color=COLORS['danger'], label="Poids Négatif (−)"),
]
ax.legend(handles=handles, loc="lower right", frameon=True, edgecolor='#E5E7EB')
plt.tight_layout()
plt.show()


## 5. Backtest de la Stratégie 1 (Recalculé en Live)
Nous simulons l'exécution du modèle de régime avec un délai $\epsilon = 1$ min et les spreads bid-ask.


In [ ]:
from backtest_engine import load_sp100_returns, strategy1_generate_signals, apply_epsilon_delay, compute_spread_costs, compute_pnl, load_quotes, buy_and_hold_benchmark

print("1. Chargement des rendements d'actions de marché...")
df_returns = load_sp100_returns(start=START_DATE, resample="15min")

print("\n2. Ré-échantillonnage des features à la fréquence de trading (15min)...")
df_poly_15m = df_features.resample("15min").last()

print("\n3. Génération des signaux de trading (Strategy 1)...")
raw_weights = strategy1_generate_signals(df_poly_15m, df_returns, period="test")

print("\n4. Exécution de la simulation de portefeuille et calcul du PnL...")
delayed_weights = apply_epsilon_delay(raw_weights, epsilon_minutes=1)
df_quotes = load_quotes()
spread_costs = compute_spread_costs(delayed_weights, df_quotes)

res_s1 = compute_pnl(delayed_weights, df_returns, spread_costs, is_causal_strat=True)
res_bh = buy_and_hold_benchmark(df_returns)

comparison_data = {
    "Strategy_1_Causal": res_s1["metrics"],
    "BuyAndHold_SP100": res_bh["metrics"]
}

df_compare = pd.DataFrame(comparison_data)
df_display = df_compare.T
df_display = df_display[[
    'annualised_gross', 'annualised_net', 'sharpe_ratio', 
    'max_drawdown', 'n_trades', 'total_spread_cost', 'avg_gross_exposure'
]].rename(columns={
    'annualised_gross': 'Retour Brut Ann.',
    'annualised_net': 'Retour Net Ann.',
    'sharpe_ratio': 'Ratio de Sharpe',
    'max_drawdown': 'Drawdown Maximum',
    'n_trades': 'Nombre de Trades',
    'total_spread_cost': 'Coût Total Spread',
    'avg_gross_exposure': 'Exposition Brute Moy.'
})

df_formatted = df_display.copy()
for col in df_display.columns:
    df_formatted[col] = [format_value(row[col], col) for idx, row in df_display.iterrows()]
    
styled_table = df_formatted.style.set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#1E3A8A'), ('color', 'white'), ('font-weight', 'bold'), ('text-align', 'left'), ('padding', '12px 16px'), ('font-family', 'sans-serif')]},
    {'selector': 'td', 'props': [('padding', '10px 16px'), ('border-bottom', '1px solid #E5E7EB'), ('font-family', 'monospace'), ('font-size', '14px')]},
    {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#F9FAFB')]},
    {'selector': 'tr:hover', 'props': [('background-color', '#EFF6FF'), ('transition', 'background-color 0.2s')]}
]).set_properties(**{
    'width': '800px',
    'border': '1px solid #E5E7EB'
})

display(HTML("<div style='margin: 20px 0;'><h3 style='color: #1E3A8A; font-family: sans-serif; font-weight: bold;'>📊 Tableau de Bord : Strategy 1 vs Buy & Hold SP100</h3></div>"))
display(styled_table)


### 5.1 Trajectoires des Rendements Nets Cumulés
Ce graphique montre l'évolution du capital net de frais de transaction de notre stratégie face au benchmark Buy & Hold.


In [ ]:
fig, ax = plt.subplots(figsize=(15, 6.5), dpi=100)

# Benchmark
cum_bh = res_bh["cum_net"]
ax.plot(cum_bh.index, cum_bh.values * 100, label="Buy & Hold SP100 (Benchmark)", 
        color=COLORS['gray'], linewidth=1.8, linestyle="--", zorder=1)

# Strategy 1
cum_s1 = res_s1["cum_net"]
ax.plot(cum_s1.index, cum_s1.values * 100, label="Net PnL — Strategy 1 Causal", 
        color=COLORS['primary'], linewidth=2.5, zorder=2)
ax.fill_between(cum_s1.index, 0, cum_s1.values * 100, alpha=0.06, color=COLORS['primary'])

ax.axhline(0, color=COLORS['gray'], linewidth=1, linestyle='--')

ax.set_title('📊 Trajectoires des Rendements Nets cumulés (après spreads de transaction)', fontsize=15, fontweight='bold', pad=15, color='#1F2937')
ax.set_ylabel('Rendement Net Cumulé (%)', fontsize=12, fontweight='semibold')
ax.set_xlabel('Date', fontsize=12, fontweight='semibold')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b\n%Y'))
ax.legend(loc='upper left', frameon=True, facecolor='white', edgecolor='#E5E7EB', fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6, color='#D1D5DB')
plt.tight_layout()
plt.show()


### 5.2 Profils de Risques : Drawdowns Temporaires (Underwater Drawdown)
Le graphique de drawdown montre les pertes maximales subies par la stratégie depuis ses sommets historiques.


In [ ]:
fig, ax = plt.subplots(figsize=(15, 5), dpi=100)

# Benchmark
dd_bh = res_bh["drawdown"]
ax.plot(dd_bh.index, dd_bh.values * 100, label="Buy & Hold SP100", 
        color=COLORS['gray'], linewidth=1.8, linestyle="--", zorder=1)

# Strategy 1
dd_s1 = res_s1["drawdown"]
ax.plot(dd_s1.index, dd_s1.values * 100, label="Drawdown — Strategy 1 Causal", 
        color=COLORS['drawdown'], linewidth=1.8, zorder=2)
ax.fill_between(dd_s1.index, 0, dd_s1.values * 100, alpha=0.08, color=COLORS['drawdown'])

ax.axhline(0, color='#9CA3AF', linewidth=1)
ax.set_title('📉 Profils de Risques : Drawdowns Temporaires (Underwater Drawdown)', fontsize=15, fontweight='bold', pad=15, color='#1F2937')
ax.set_ylabel('Drawdown (%)', fontsize=12, fontweight='semibold')
ax.set_xlabel('Date', fontsize=12, fontweight='semibold')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b\n%Y'))
ax.legend(loc='lower left', frameon=True, facecolor='white', edgecolor='#E5E7EB', fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6, color='#D1D5DB')
plt.tight_layout()
plt.show()


### 5.3 Analyse de Risque Glissante (Sharpe et Volatilité mobile de 7j)
Suivi dynamique de la performance ajustée au risque.


In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 9), sharex=True, dpi=100)

window = 182  # ~7 jours à 15m
ppy = 6552    # périodes par an

# Strategy 1
nr = res_s1["net_returns"]
rmean = nr.rolling(window).mean()
rstd  = nr.rolling(window).std()
rvol  = rstd * np.sqrt(ppy) * 100
rsharpe = (rmean / rstd * np.sqrt(ppy)).where(rstd > 0, 0)

# Plot Sharpe
ax1.plot(rsharpe.index, rsharpe.values, color=COLORS['primary'], linewidth=2.2, label="Sharpe — Strategy 1")
ax1.axhline(0, color='grey', linestyle='--', linewidth=0.8)
ax1.set_title('⚖️ Ratio de Sharpe Glissant Glissant (7j)', fontsize=13, fontweight='bold')
ax1.set_ylabel('Ratio de Sharpe', fontsize=11)
ax1.legend(loc='upper left', frameon=True, facecolor='white', edgecolor='#E5E7EB', fontsize=9)
ax1.grid(True, alpha=0.3)

# Plot Volatility
ax2.plot(rvol.index, rvol.values, color=COLORS['warning'], linewidth=2.2, label="Volatilité — Strategy 1")
ax2.set_title('⚡ Volatilité Annualisée Glissante (7j)', fontsize=13, fontweight='bold')
ax2.set_ylabel('Volatilité (%)', fontsize=11)
ax2.set_xlabel('Date', fontsize=12)
ax2.legend(loc='upper left', frameon=True, facecolor='white', edgecolor='#E5E7EB', fontsize=9)
ax2.grid(True, alpha=0.3)

ax2.xaxis.set_major_formatter(mdates.DateFormatter('%d %b\n%Y'))
plt.tight_layout()
plt.show()


### 5.4 Exposition Long vs Short Cumulée du Portefeuille
Ce graphique illustre comment le capital est réparti entre les positions acheteuses (Long) et vendeuses (Short) pour chaque actif.


In [ ]:
fig, ax = plt.subplots(figsize=(15, 6.5), dpi=100)

weights = res_s1['weights']
active_cols = [c for c in weights.columns if (weights[c] != 0).any()]

if active_cols:
    daily_weights = weights[active_cols].resample('D').mean()
    pos_w = daily_weights.clip(lower=0)
    neg_w = daily_weights.clip(upper=0)
    
    cmap = plt.get_cmap('tab20', len(active_cols))
    colors = [cmap(i) for i in range(len(active_cols))]
    
    ax.stackplot(daily_weights.index, pos_w.T, colors=colors, alpha=0.8, labels=active_cols)
    ax.stackplot(daily_weights.index, neg_w.T, colors=colors, alpha=0.8)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title("Exposition Cumulative du Portefeuille (Long vs Short)", fontweight='bold', fontsize=14, pad=15)
    ax.set_ylabel('Poids du Portefeuille', fontsize=12, fontweight='semibold')
    ax.set_xlabel('Date', fontsize=12, fontweight='semibold')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper left', ncol=6, frameon=True, edgecolor='#E5E7EB', facecolor='white', fontsize=9)
else:
    ax.text(0.5, 0.5, "Aucune position active", ha='center', va='center', fontsize=14)
    ax.set_title("Exposition Cumulative du Portefeuille", fontweight='bold')

plt.tight_layout()
plt.show()


### 5.5 Carte Thermique d'Exposition (Rotation des Actifs)
Ce graphique montre comment les poids de portefeuille varient dans le temps sur chaque actif individuel.


In [ ]:
fig, ax = plt.subplots(figsize=(15, 6.5), dpi=100)

weights = res_s1['weights']
daily_weights = weights.resample('D').mean()
active_cols = [c for c in daily_weights.columns if (daily_weights[c] != 0).any()]

if active_cols:
    heatmap_data = daily_weights[active_cols].T
    sns.heatmap(heatmap_data, cmap='RdBu', center=0, cbar=True, ax=ax, 
                linewidths=0.05, linecolor='#F3F4F6', cbar_kws={'label': 'Poids'})
    
    ax.set_title("Carte Thermique d'Exposition Temporelle (Rotation des Actifs)", fontweight='bold', fontsize=14, pad=15)
    ax.set_xlabel('Date', fontsize=12, fontweight='semibold')
    ax.set_ylabel('Actif', fontsize=12, fontweight='semibold')
    
    # Formater l'axe des dates
    step = max(1, heatmap_data.shape[1] // 8)
    x_indices = list(range(0, heatmap_data.shape[1], step))
    ax.set_xticks([x + 0.5 for x in x_indices])
    ax.set_xticklabels([heatmap_data.columns[x].strftime('%d %b') for x in x_indices], rotation=0)
else:
    ax.text(0.5, 0.5, "Aucune position active", ha='center', va='center', fontsize=14)
    ax.set_title("Carte Thermique d'Exposition", fontweight='bold')

plt.tight_layout()
plt.show()


### 5.6 Profil Statistique des Rendements Actifs
L'analyse de distribution des rendements actifs non nuls (exprimés en points de base - bps) permet d'observer l'espérance de gain moyenne et les anomalies (asymétrie, aplatissement).


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), dpi=100)

net_returns = res_s1['net_returns']
active_returns = net_returns[net_returns != 0] * 10000  # Conversion en points de base (bps)

if not active_returns.empty:
    sns.histplot(active_returns, kde=True, color=COLORS['accent'], bins=40, ax=ax, stat='density', alpha=0.35)
    
    mean_val = active_returns.mean()
    ax.axvline(mean_val, color='blue', linestyle='--', linewidth=1.2, label=f'Moy: {mean_val:+.1f} bps')
    ax.axvline(0, color='black', linewidth=0.8)
    
    skew = active_returns.skew()
    kurt = active_returns.kurtosis()
    stats_box = f"Trades: {len(active_returns)}\nMoy: {mean_val:+.1f} bps\nStd: {active_returns.std():.1f} bps\nSkew: {skew:+.2f}\nKurt: {kurt:.2f}"
    
    ax.text(0.95, 0.95, stats_box, transform=ax.transAxes, verticalalignment='top',
            horizontalalignment='right',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='#E5E7EB', alpha=0.85),
            fontsize=9, fontfamily='monospace')
    
    ax.set_title("Distribution Statistique des Rendements Actifs Périodiques (bps)", fontweight='bold', fontsize=14, pad=15)
    ax.set_xlabel('Rendement Actif (bps)', fontsize=12, fontweight='semibold')
    ax.set_ylabel('Densité', fontsize=12, fontweight='semibold')
else:
    ax.text(0.5, 0.5, "Aucun rendement actif non-nul", ha='center', va='center', fontsize=14)
    ax.set_title("Distribution des Rendements", fontweight='bold')

plt.tight_layout()
plt.show()
